# Nettoyage des données

- Suppression des colonnes inutiles ou trop vides (`_score`, `typologie_logement`, etc.)
- Imputation des valeurs manquantes (médiane ou "NC")
- Normalisation des types (float, string)
- Nettoyage des valeurs aberrantes (surfaces, consommations, GES, etc.)
- Vérification des coordonnées géographiques et des bornes temporelles
- Export du fichier propre : `../data/df_adem_cleaned_69.csv`

### 01. Imports

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Charger les données fusionnées
df = pd.read_csv("../data/df_adem_merge_69.csv", low_memory=False)

print(f"{len(df)} lignes – {len(df.columns)} colonnes")
df.head()

445289 lignes – 39 colonnes


,conso_5_usages_ep,emission_ges_5_usages,nom_commune_ban,type_logement_source,cout_total_5_usages,etiquette_ges,methode_application_dpe,type_energie_principale_chauffage,_score,code_region_ban,...,typologie_logement,isolation_toiture,etiquette_dpe,coordonnee_cartographique_x_ban,code_postal_ban,zone_climatique,presence_production_pv,qualite_isolation_menuiseries,date_reception_dpe,emission_ges_chauffage
0,28727.8,8507.0,Saint-Clément-de-Vers,existant,2693.3,E,dpe maison individuelle,Fioul domestique,NaN,84.0,...,NaN,1.0,E,807850.62,69790,H1c,NaN,moyenne,2022-05-14,7671.2
1,47440.8,6325.7,Propières,existant,2860.4,E,dpe maison individuelle,Bois – Bûches,NaN,84.0,...,NaN,NaN,G,811688.48,69790,H1c,NaN,moyenne,2022-01-18,6136.6
2,41945.9,1379.9,Saint-Bonnet-des-Bruyères,existant,2619.4,C,dpe maison individuelle,Électricité,NaN,84.0,...,NaN,NaN,G,815715.28,69790,H1c,1.0,insuffisante,2021-11-24,1286.4
3,19626.9,646.9,Saint-Bonnet-des-Bruyères,existant,1743.0,A,dpe maison individuelle,Électricité,NaN,84.0,...,NaN,NaN,C,814019.46,69790,H1c,NaN,très bonne,2021-08-21,517.7
4,56620.0,16442.9,Propières,existant,5134.6,G,dpe maison individuelle,Électricité,NaN,84.0,...,NaN,NaN,G,812372.97,69790,H1c,1.0,insuffisante,2021-11-30,16260.5


### 02. Inspection des types et des valeurs manquantes

In [3]:
df.info()

missing_ratio = df.isna().mean().sort_values(ascending=False)
print("Pourcentage de valeurs manquantes (%):")
print((missing_ratio * 100).round(1).head(15))


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 445289 entries, 0 to 445288
Data columns (total 39 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   conso_5_usages_ep                  445281 non-null  float64
 1   emission_ges_5_usages              445281 non-null  float64
 2   nom_commune_ban                    445289 non-null  object 
 3   type_logement_source               445289 non-null  object 
 4   cout_total_5_usages                445278 non-null  float64
 5   etiquette_ges                      445289 non-null  object 
 6   methode_application_dpe            445289 non-null  object 
 7   type_energie_principale_chauffage  445289 non-null  object 
 8   _score                             0 non-null       float64
 9   code_region_ban                    444830 non-null  float64
 10  conso_refroidissement_ep           445289 non-null  float64
 11  qualite_isolation_murs             4452

In [4]:
cols_to_drop = ["_score", "typologie_logement","type_ventilation","presence_production_pv"]
df = df.drop(columns=cols_to_drop)

### 03. Imputation des valeurs manquantes

In [5]:
# Numériques : médiane
num_median = [
    "inertie_lourde", "isolation_toiture", "annee_construction",
    "nombre_niveau_logement", "surface_habitable_logement"
]
for col in num_median:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

# Numériques à imputation simple (mode)
num_mode = ["code_region_ban", "code_departement_ban"]
for col in num_mode:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].mode()[0])

# Catégorielles à imputer par "NC"
cat_nc = ["type_installation_chauffage", "type_installation_ecs", "zone_climatique"]
for col in cat_nc:
    if col in df.columns:
        df[col] = df[col].fillna("NC")

# Très faibles taux de NaN (supprimer les lignes concernées)
df = df.dropna(subset=[
    "conso_5_usages_ep", "emission_ges_5_usages", "cout_total_5_usages",
    "emission_ges_chauffage", "conso_chauffage_ep"
])

# Catégorielle à imputer par mode
if "qualite_isolation_murs" in df.columns:
    df["qualite_isolation_murs"] = df["qualite_isolation_murs"].fillna(df["qualite_isolation_murs"].mode()[0])


In [6]:
missing_final = df.isna().mean().sort_values(ascending=False)
print("Pourcentage de NaN restants après imputation :")
print((missing_final * 100).round(2).head(10))

Pourcentage de NaN restants après imputation :
conso_5_usages_ep              0.0
conso_chauffage_ep             0.0
modele_dpe                     0.0
type_installation_ecs          0.0
code_departement_ban           0.0
surface_habitable_logement     0.0
type_installation_chauffage    0.0
hauteur_sous_plafond           0.0
isolation_toiture              0.0
nombre_niveau_logement         0.0
dtype: float64


In [7]:
print(f"{len(df)} lignes – {len(df.columns)} colonnes")

445278 lignes – 35 colonnes


### 04. Normalisation des types + Nettoyage des valeurs aberrantes

In [8]:
# --- Colonnes numériques à normaliser ---
numeric_cols = [
    "surface_habitable_logement",
    "hauteur_sous_plafond",
    "conso_5_usages_ep",
    "conso_chauffage_ep",
    "conso_ecs_ep",
    "conso_refroidissement_ep",
    "emission_ges_5_usages",
    "emission_ges_chauffage",
    "cout_total_5_usages",
    "annee_construction",
    "nombre_niveau_logement",
    "inertie_lourde",
    "isolation_toiture",
    "code_region_ban",
    "code_departement_ban",
    "coordonnee_cartographique_x_ban",
    "coordonnee_cartographique_y_ban"
]

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# --- Colonnes catégorielles à normaliser ---
cat_cols = [
    "etiquette_dpe",
    "etiquette_ges",
    "type_batiment",
    "methode_application_dpe",
    "type_energie_principale_chauffage",
    "type_installation_chauffage",
    "type_installation_ecs",
    "zone_climatique",
    "qualite_isolation_murs",
    "qualite_isolation_menuiseries",
    "nom_commune_ban",
    "type_logement_source"
]

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.upper()

#### Nettoyage des valeurs aberrantes

In [9]:
df[numeric_cols].describe()

,surface_habitable_logement,hauteur_sous_plafond,conso_5_usages_ep,conso_chauffage_ep,conso_ecs_ep,conso_refroidissement_ep,emission_ges_5_usages,emission_ges_chauffage,cout_total_5_usages,annee_construction,nombre_niveau_logement,inertie_lourde,isolation_toiture,code_region_ban,code_departement_ban,coordonnee_cartographique_x_ban,coordonnee_cartographique_y_ban
count,445278.000000,445278.000000,4.452780e+05,4.452780e+05,4.452780e+05,445278.000000,4.452780e+05,4.452780e+05,4.452780e+05,445278.000000,445278.000000,445278.000000,445278.000000,445278.0,445278.000000,445278.000000,4.452780e+05
mean,68.313554,2.612538,1.962153e+04,1.330066e+04,4.618228e+03,28.340183,2.607009e+03,2.022270e+03,1.698974e+03,1981.791068,1.295620,0.193937,0.207178,84.0,68.999930,840818.782377,6.512155e+06
std,69.776494,3.678803,4.642955e+05,4.593960e+05,1.775655e+04,628.468006,1.044071e+05,1.041099e+05,2.892225e+04,25.581971,1.401777,0.395381,0.405285,0.0,0.046456,30925.499537,2.323155e+05
min,1.000000,0.200000,3.035000e+02,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,3.830000e+01,1460.000000,1.000000,0.000000,0.000000,84.0,38.000000,0.000000,0.000000e+00
25%,45.000000,2.500000,6.819125e+03,3.368000e+03,1.927100e+03,0.000000,4.409000e+02,2.540250e+02,6.434000e+02,1972.000000,1.000000,0.000000,0.000000,84.0,69.000000,839710.700000,6.516713e+06
50%,63.200000,2.500000,1.074350e+04,6.484150e+03,2.713400e+03,0.000000,1.042000e+03,6.624000e+02,9.500000e+02,1980.000000,1.000000,0.000000,0.000000,84.0,69.000000,843181.620000,6.519380e+06
75%,80.000000,2.500000,1.661050e+04,1.183658e+04,4.016700e+03,0.000000,2.212400e+03,1.647600e+03,1.405400e+03,1994.000000,1.000000,0.000000,0.000000,84.0,69.000000,845817.657500,6.521600e+06
max,11298.800000,2049.000000,3.067888e+08,3.051387e+08,1.557491e+06,214948.800000,6.932379e+07,6.926649e+07,1.892502e+07,2025.000000,306.000000,1.000000,1.000000,84.0,69.000000,865916.300000,6.579322e+06


##### Consommations énergétiques

In [10]:
df = df[
    (df["conso_5_usages_ep"].between(100, 100_000)) &
    (df["conso_chauffage_ep"].between(0, 80_000)) &
    (df["conso_ecs_ep"].between(0, 10_000)) &
    (df["conso_refroidissement_ep"].between(0, 5_000))
]

##### Émissions de gaz à effet de serre (GES)

In [11]:
df = df[
    (df["emission_ges_5_usages"].between(0, 10_000)) &
    (df["emission_ges_chauffage"].between(0, 10_000))
]

##### Coût total énergétique

In [12]:
df = df[(df["cout_total_5_usages"] > 0) & (df["cout_total_5_usages"] < 20_000)]

##### Surface & caractéristiques bâtiment

In [13]:
df = df[
    (df["surface_habitable_logement"].between(9, 1_000)) &
    (df["hauteur_sous_plafond"].between(2, 5)) &
    (df["nombre_niveau_logement"].between(1, 10))
]

#### Vérification post-filtrage

In [14]:
df.describe()[["surface_habitable_logement","conso_5_usages_ep","emission_ges_5_usages","annee_construction"]].T

,count,mean,std,min,25%,50%,75%,max
surface_habitable_logement,427661.0,65.540993,33.364205,9.0,44.2,63.0,79.8,604.8
conso_5_usages_ep,427661.0,12503.638596,8436.508141,303.5,6695.0,10455.5,15883.9,93511.8
emission_ges_5_usages,427661.0,1512.356398,1530.657752,0.0,432.6,1000.6,2067.2,9999.9
annee_construction,427661.0,1982.031476,25.603212,1460.0,1973.0,1980.0,1994.0,2025.0


In [15]:
print(f"{len(df)} lignes – {len(df.columns)} colonnes")

427661 lignes – 35 colonnes


In [16]:
df.to_csv("../data/df_adem_cleaned_69.csv", index=False, encoding="utf-8")
print("Fichier fusionné enregistré : ../data/df_adem_cleaned_69.csv")

Fichier fusionné enregistré : ../data/df_adem_cleaned_69.csv
